# Phishing URL Detector - Simple Demo
Full ML Pipeline: Feature Extraction -> Dataset -> Training -> Evaluation -> Prediction

---
**Model:** Random Forest Classifier (100 trees)  
**Dataset:** 60 URLs (30 Legitimate + 30 Phishing)  
**Features:** 20 URL-based features


## Install Dependencies

In [ ]:
!pip install scikit-learn pandas --quiet
print("All dependencies ready!")

---
## STEP 1 - Imports
Import all libraries needed.


In [ ]:
import re
import math
from collections import Counter
from urllib.parse import urlparse

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report
)

print("Imports done!")

---
## STEP 2 - Feature Extractor
We look at **20 things (features)** about a URL to decide if it is phishing or legitimate.

Key features we extract:
- `url_length` - total number of characters
- `has_https` - 1 if URL uses HTTPS, 0 if not
- `has_ip_address` - 1 if an IP is used instead of a domain
- `has_at_symbol` - 1 if @ is present (hides the real host)
- `has_shortener` - 1 if bit.ly / tinyurl etc. is used
- `num_suspicious_keywords` - count of words like login, verify, secure
- `has_brand_in_subdomain_or_path` - 1 if a brand appears outside its real domain
- `url_entropy` - how random/scrambled the URL characters look
- `tld` - top-level domain (.com, .xyz, .tk etc.)


In [ ]:
# Regex to detect IP addresses like 192.168.1.1
IP_PATTERN = re.compile(r"^\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}$")

# Known URL shorteners
SHORTENERS = ["bit.ly", "tinyurl.com", "goo.gl", "t.co", "ow.ly", "is.gd"]

# Words commonly seen in phishing URLs
SUSPICIOUS_KEYWORDS = [
    "login", "signin", "verify", "update", "banking",
    "secure", "account", "confirm", "password", "suspend",
    "alert", "support", "validate", "unlock", "billing"
]

# Brands phishers commonly impersonate
TARGET_BRANDS = [
    "google", "netflix", "facebook", "microsoft", "apple",
    "amazon", "paypal", "instagram", "chase", "sbi",
    "irctc", "wellsfargo", "hdfc", "icici", "whatsapp"
]


def extract_features(raw_url):
    """Given a URL string, return a dictionary of 20 features."""

    # Add http:// if missing so urlparse works
    url = raw_url.strip()
    if not url.startswith(("http://", "https://")):
        url = "http://" + url

    parsed = urlparse(url)

    # Get clean hostname (remove @ user info and port)
    host = parsed.netloc.split("@")[-1].split(":")[0].strip().lower()
    host_parts = host.split(".") if host else []
    is_ip = bool(IP_PATTERN.match(host))

    num_digits = sum(c.isdigit() for c in raw_url)
    url_len = len(raw_url.strip())

    # Shannon entropy: high value means URL looks random/obfuscated
    if url_len > 0:
        freq = Counter(raw_url)
        entropy = -sum((c/url_len)*math.log2(c/url_len) for c in freq.values())
    else:
        entropy = 0.0

    found_kw = [kw for kw in SUSPICIOUS_KEYWORDS if kw in url.lower()]

    # Brand spoofing check
    has_brand = 0
    for brand in TARGET_BRANDS:
        if brand in url.lower():
            if not (host == f"{brand}.com" or host.endswith(f".{brand}.com")):
                has_brand = 1
                break

    tld = "ip" if is_ip else (host_parts[-1] if host_parts else "")

    return {
        "url_length":                     url_len,
        "hostname_length":                len(host),
        "path_length":                    len(parsed.path),
        "num_dots":                       raw_url.count("."),
        "num_hyphens":                    raw_url.count("-"),
        "num_underscores":                raw_url.count("_"),
        "num_digits":                     num_digits,
        "num_special_chars":              sum(not c.isalnum() for c in raw_url),
        "num_subdomains":                 max(0, len(host_parts)-2) if not is_ip else 0,
        "url_depth":                      len([s for s in parsed.path.split("/") if s]),
        "num_query_params":               len(parsed.query.split("&")) if parsed.query else 0,
        "has_at_symbol":                  1 if "@" in raw_url else 0,
        "has_https":                      1 if parsed.scheme == "https" else 0,
        "has_ip_address":                 1 if is_ip else 0,
        "has_shortener":                  1 if any(s in host for s in SHORTENERS) else 0,
        "num_suspicious_keywords":        len(found_kw),
        "has_brand_in_subdomain_or_path": has_brand,
        "digit_ratio":                    round(num_digits/url_len, 4) if url_len > 0 else 0.0,
        "url_entropy":                    round(entropy, 4),
        "tld":                            tld,
    }


# Quick test on a sample phishing URL
sample = extract_features("http://paypal-secure-login.tk/verify-account")
print("Sample features for a phishing URL:")
for k, v in sample.items():
    print(f"  {k:40s}: {v}")

---
## STEP 3 - Dataset
We label 30 legitimate and 30 phishing URLs, extract features from each, and build a DataFrame.
- **Label 0** = Legitimate
- **Label 1** = Phishing


In [ ]:
df = build_dataset()
print(f"Total URLs  : {len(df)}")
print(f"Legitimate  : {(df['label'] == 0).sum()}")
print(f"Phishing    : {(df['label'] == 1).sum()}")
print(f"Features    : {df.shape[1] - 1}")
print()
df.head(10)


---
## STEP 4 - Model Training
We train a **Random Forest Classifier** (ensemble of 100 decision trees).

Pipeline:
1. **OneHotEncoder** - converts the `tld` text column to numbers
2. **RandomForestClassifier** - 100 trees vote on the final answer

Split: **80% training / 20% testing** (stratified so both halves have equal class ratio).


In [ ]:
def build_dataset():
    # 30 Legitimate URLs (Label 0)
    legitimate_urls = [
        "https://www.google.com", "https://www.github.com", "https://www.amazon.com",
        "https://www.wikipedia.org", "https://www.microsoft.com", "https://www.apple.com",
        "https://www.netflix.com", "https://www.facebook.com", "https://www.instagram.com",
        "https://www.twitter.com", "https://www.linkedin.com", "https://www.youtube.com",
        "https://www.reddit.com", "https://www.medium.com", "https://www.stackoverflow.com",
        "https://www.nytimes.com", "https://www.cnn.com", "https://www.bbc.co.uk",
        "https://www.espn.com", "https://www.weather.com", "https://www.imdb.com",
        "https://www.dropbox.com", "https://www.spotify.com", "https://www.zoom.us",
        "https://www.salesforce.com", "https://www.quora.com", "https://www.tumblr.com",
        "https://www.pinterest.com", "https://www.ebay.com", "https://www.craigslist.org"
    ]

    # 30 Phishing URLs (Label 1)
    phishing_urls = [
        "http://paypal-secure-login.tk/verify-account",
        "http://192.168.1.1/banking/login.html",
        "http://secure-login.apple.com.phishing-site.xyz/login",
        "http://bit.ly/3xPhish1ng",
        "http://hdfc-net-banking-secure.xyz/login",
        "http://secure-chase-update.info",
        "http://signin.netflix-billing-alert.cc",
        "http://verify-facebook-profile.tk",
        "http://wellsfargo-secure-banking.net/signin",
        "http://amazon-giftcard-winner.xyz",
        "http://google-account-verify.online",
        "http://instagram-login-verify.top",
        "http://microsoft-support-alert.club",
        "http://sbi-online-banking-confirm.site",
        "http://irctc-ticket-refund.xyz",
        "http://whatsapp-security-update.tk",
        "http://icici-net-banking-verify.info",
        "http://paypal-customer-service.club",
        "http://apple-id-suspended-alert.net",
        "http://chase-bank-verify-account.top",
        "http://10.22.44.11/secure-login/",
        "http://tinyurl.com/y4phish23",
        "http://is.gd/securebanking",
        "http://ow.ly/fakeupdate",
        "http://billing-update-netflix.tk",
        "http://secure-login-instagram.cf",
        "http://validate-your-account-now.gq",
        "http://unlock-your-profile-here.ga",
        "http://ebay-member-security-update.club",
        "http://facebook-security-check.xyz"
    ]

    data = []
    for url in legitimate_urls:
        features = extract_features(url)
        features["label"] = 0
        data.append(features)

    for url in phishing_urls:
        features = extract_features(url)
        features["label"] = 1
        data.append(features)

    return pd.DataFrame(data)

df = build_dataset()
print(f"Total URLs  : {len(df)}")
print(f"Legitimate  : {(df['label'] == 0).sum()}")
print(f"Phishing    : {(df['label'] == 1).sum()}")
print(f"Features    : {df.shape[1] - 1}")
print()
df.head(10)

---
## STEP 5 - Evaluation
We test the model on the 20% of URLs it never saw during training.

- **Accuracy** = % of all predictions that were correct
- **Precision** = of URLs flagged as phishing, how many really were?
- **Recall** = of all real phishing URLs, how many did we catch?
- **F1-Score** = balance of Precision and Recall
- **ROC-AUC** = overall ranking ability (1.0 = perfect)


In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec  = recall_score(y_test, y_pred, zero_division=0)
f1   = f1_score(y_test, y_pred, zero_division=0)
roc  = roc_auc_score(y_test, y_prob)
cm   = confusion_matrix(y_test, y_pred)

print("=" * 50)
print("  Evaluation Results")
print("=" * 50)
print(f"  Accuracy   : {acc * 100:.2f}%")
print(f"  Precision  : {prec:.4f}")
print(f"  Recall     : {rec:.4f}")
print(f"  F1-Score   : {f1:.4f}")
print(f"  ROC-AUC    : {roc:.4f}")
print()
print("  Confusion Matrix:")
print(f"              Predicted Legit   Predicted Phish")
print(f"  Actual Legit       {cm[0][0]:4d}              {cm[0][1]:4d}")
print(f"  Actual Phish       {cm[1][0]:4d}              {cm[1][1]:4d}")
print()
print("  Classification Report:")
print(classification_report(y_test, y_pred, target_names=["Legitimate", "Phishing"]))

---
## STEP 6 - Live Prediction
Given any URL, we extract its 20 features and run the trained model.
The result also shows **why** the URL was flagged using risk signals.


In [ ]:
def predict_url(model, url):
    features = extract_features(url)
    df_input = pd.DataFrame([features])
    prediction = model.predict(df_input)[0]
    proba = model.predict_proba(df_input)[0]
    label = "PHISHING" if prediction == 1 else "LEGITIMATE"
    phish_pct = round(proba[1] * 100, 1)
    legit_pct = round(proba[0] * 100, 1)
    risk = []
    if features["has_ip_address"]:              risk.append("IP address as host")
    if features["has_at_symbol"]:               risk.append("@ symbol in URL")
    if features["has_shortener"]:               risk.append("URL shortener used")
    kw = features["num_suspicious_keywords"]
    if kw > 0:                                   risk.append(str(kw) + " suspicious keyword(s)")
    if features["has_brand_in_subdomain_or_path"]:risk.append("Brand spoofed in URL")
    if not features["has_https"]:               risk.append("No HTTPS")
    return {"url": url, "label": label,
            "phish_pct": phish_pct, "legit_pct": legit_pct, "risk": risk}


def print_prediction(r):
    signals = ", ".join(r["risk"]) if r["risk"] else "None - looks clean"
    print("-" * 58)
    print("  Result      : " + r["label"])
    print("  URL         : " + r["url"][:65])
    print("  Phishing %  : " + str(r["phish_pct"]) + "%")
    print("  Legit    %  : " + str(r["legit_pct"]) + "%")
    print("  Risk Signals: " + signals)


print("Prediction functions ready!")

### Demo - Batch Predictions
Testing the model on known phishing and legitimate URLs.


In [ ]:
demo_urls = [
    "http://paypal-secure-login.tk/verify-account",
    "http://192.168.1.1/banking/login.html",
    "http://secure-login.apple.com.phishing-site.xyz/login",
    "http://bit.ly/3xPhish1ng",
    "https://www.google.com",
    "https://www.github.com",
    "https://www.amazon.com",
    "https://www.wikipedia.org",
]

for url in demo_urls:
    r = predict_url(model, url)
    print_prediction(r)

---
## STEP 7 - Try Your Own URL
Change the URL below and run the cell to check any link!


In [ ]:
# Change this URL to anything you want to check
url_to_check = "http://hdfc-net-banking-secure.xyz/login"

r = predict_url(model, url_to_check)
print_prediction(r)